# Forward-Secure Lattice-Based IBE for Internet of Things

**Reference:** *A lattice-based forward secure IBE scheme for Internet of things* (research paper implementation).

---

## 1. Overview

This notebook implements a **forward-secure Identity-Based Encryption (fs-IBE)** scheme built on **lattice assumptions**, designed for IoT settings where:

- **Identity-based encryption (IBE):** Encryption uses a user’s identity (e.g. string ID) as the public key; no certificates are needed.
- **Forward security:** Secret keys are updated over time (epochs). Compromise of the key at time $t$ does not allow decryption of ciphertexts from earlier epochs.
- **Lattice-based cryptography:** Security relies on the hardness of lattice problems (e.g. SIS/LWE), giving **post-quantum security**.
- **Trust model:** Queries are authenticated (e.g. via signatures) and tied to a trust/reputation system to mitigate malicious or abusive queries.

---

## 2. What This Implementation Does

| Part | Description |
|------|-------------|
| **P1 — Lattice Infrastructure** | Trapdoor generation (TrapGen), preimage sampling (SamplePre), gadget matrix, binary tree for epochs, **Setup** and **KeyGen** (identity → secret key). |
| **P2 — Forward Security** | **Encrypt** / **Decrypt** (Dual Regev–style IBE), **Key Update** (minimal cover over epochs), epoch-based encryption. |
| **P3 — Trust Model** | Signature stub (Dilithium-style), **TrustManager** (reward/penalize), **Query** + **QueryValidator** (authenticated, trust-checked queries). |
| **Table 1** | fs-IBE parameter sets: **PARA.512**, **PARA.768**, **PARA.1024** (n, q, δ, NIST level, bits of security). |
| **P4 — Simulation** | Full pipeline: Setup → encrypt data → run queries (encrypt, trust check, match, decrypt) → measure latency/throughput and **False Trust Acceptance Rate (FTAR)**; outputs **Results_Report.csv**. |

**Notation (paper-aligned):**

- $n$: lattice dimension; $q$: modulus; $\sigma$: Gaussian width.
- **A**: public matrix (with trapdoor **T_A**); **u** = $H(\mathsf{id})$: identity vector; **sk**: secret key (short vector s.t. **A**·**sk** = **u** mod $q$).
- Epochs $t = 0, 1, \ldots, T-1$ with binary tree; **minimal cover** for time $t$ gives the set of node keys needed to decrypt at that epoch.

---

## 3. How to Run

**Run all cells from top to bottom.** Later sections (P2, P3, P4) depend on P1; P4 uses P1–P3 and Table 1 and writes **Results_Report.csv** (or **Results_Report_output.csv** if the default file is locked).

**Full pipeline (message in → result out):** In the **Full Pipeline** section, run the code cell. You will be **prompted in the terminal/console** to enter your message. Type the message and press Enter; the notebook will then encrypt it, decrypt it, and print the full result: original message, ciphertext info, decrypted message, parameters, timings, and verification.

---
## P1 — Lattice Infrastructure

This section implements the **lattice primitives** used by the fs-IBE scheme (from `lattice_infrastructure.py`).

### 1.1 Role in the Paper

- **Lattice parameters:** Dimension $n$, modulus $q$, Gaussian parameter $\sigma$. We use $k = \lceil \log_2 q \rceil$ and $m = n \cdot k$ (column dimension for the gadget part).
- **Gadget matrix $\mathbf{G}$:** $\mathbf{G} \in \mathbb{Z}_q^{n \times nk}$ with powers-of-two structure so that any $\mathbf{u} \in \mathbb{Z}_q^n$ can be expressed as $\mathbf{G} \cdot \mathbf{y}$ for a short binary vector $\mathbf{y}$ (bit decomposition).
- **Trapdoor generation (TrapGen):** Produces a random-looking matrix $\mathbf{A} = [\bar{\mathbf{A}} \mid \mathbf{G}] \in \mathbb{Z}_q^{n \times 2m}$ and a trapdoor $\mathbf{T}_{\mathbf{A}}$ such that short preimages can be sampled.
- **Preimage sampling (SamplePre):** Given $\mathbf{A}$, $\mathbf{T}_{\mathbf{A}}$, and target $\mathbf{u}$, samples short $\mathbf{e}$ with $\mathbf{A} \cdot \mathbf{e} \equiv \mathbf{u} \pmod{q}$. Used for **KeyGen**: identity $\mathsf{id}$ is hashed to $\mathbf{u} = H(\mathsf{id})$; secret key is $\mathbf{sk}$ such that $\mathbf{A} \cdot \mathbf{sk} = \mathbf{u}$.
- **Binary tree:** Used for **forward security**. Epochs $0, \ldots, 2^d - 1$ correspond to leaves; internal nodes represent key “blobs” that are updated over time. Minimal cover (see P2) determines which node keys are needed for a given time $t$.
- **Setup:** Runs TrapGen, builds the tree; outputs public parameters (params, $\mathbf{A}$, $\mathbf{T}_{\mathbf{A}}$, tree).
- **KeyGen(system, user_id):** Computes $\mathbf{u} = H(\mathsf{user\_id})$, returns $\mathbf{sk} = \mathsf{SamplePre}(\mathbf{A}, \mathbf{T}_{\mathbf{A}}, \mathbf{u})$.

### 1.2 Implementation Notes

- **G_vector(data, params):** Hash-to-$\mathbb{Z}_q^n$: hashes `data` (e.g. identity or identity\|epoch) to a vector in $\mathbb{Z}_q^n$ (deterministic, used as public “identity vector”).
- **T_A:** Here we use a simple pedagogical trapdoor (block structure) so that SamplePre can solve $\mathbf{A} \mathbf{e} = \mathbf{u}$ via bit decomposition of $\mathbf{u}$ and the $\mathbf{G}$ part; a full implementation would use a proper trapdoor (e.g. G-trapdoor) and Gaussian sampling.
- The last block in this cell registers these primitives as `LatticeCrypto` / `lattice_infrastructure` so that P2 and P3 can import them, and runs a **unit test**: Setup → KeyGen for "Alice" → check $\mathbf{A} \cdot \mathbf{sk} \equiv \mathbf{u} \pmod{q}$.

In [7]:
import numpy as np
import hashlib
import sys
import types
from math import ceil, log2

class LatticeParams:
    def __init__(self, n=16, q=3329, sigma=3.2):
        self.n = n
        self.q = q
        self.k = ceil(log2(q))
        self.m = n * self.k
        self.sigma = sigma

def gadget_matrix(n, q):
    k = ceil(log2(q))
    G = np.zeros((n, n * k), dtype=int)
    for i in range(n):
        for j in range(k):
            G[i, i * k + j] = 1 << j
    return G % q

def bit_decompose(vec, q):
    k = ceil(log2(q))
    bits = []
    for v in vec % q:
        for i in range(k):
            bits.append((v >> i) & 1)
    return np.array(bits, dtype=int)

def TrapGen(params):
    n, q, m = params.n, params.q, params.m
    A_bar = np.random.randint(0, q, size=(n, m))
    G = gadget_matrix(n, q)
    A = np.hstack([A_bar, G]) % q
    T_A = np.vstack([np.zeros((m, m), dtype=int), np.eye(m, dtype=int)])
    return A, T_A

def discrete_gaussian(shape, sigma):
    return np.round(np.random.normal(0, sigma, size=shape)).astype(int)

def SamplePre(A, T_A, u, params):
    q = params.q
    _, m2 = A.shape
    m = m2 // 2
    y = bit_decompose(u, q)
    e = np.zeros(2 * m, dtype=int)
    e[m:] = y
    return e % q

def G_vector(data, params):
    vec = []
    ctr = 0
    while len(vec) < params.n:
        h = hashlib.sha256((data + str(ctr)).encode()).digest()
        vec.extend(h[:min(32, params.n - len(vec))])
        ctr += 1
    return np.array(vec[:params.n], dtype=int) % params.q

class BinaryTreeNode:
    def __init__(self, label):
        self.label = label
        self.left = None
        self.right = None

class BinaryTree:
    def __init__(self, depth):
        self.depth = depth
        self.root = self._build(0, 2**depth - 1)
    def _build(self, l, r):
        if l > r:
            return None
        mid = (l + r) // 2
        node = BinaryTreeNode(mid)
        node.left = self._build(l, mid - 1)
        node.right = self._build(mid + 1, r)
        return node

def Setup(tree_depth=4, params=None):
    if params is None:
        params = LatticeParams()
    A, T_A = TrapGen(params)
    return {"params": params, "A": A, "T_A": T_A, "tree": BinaryTree(tree_depth)}

def KeyGen(system, user_id):
    params = system["params"]
    u = G_vector(user_id, params)
    return SamplePre(system["A"], system["T_A"], u, params)

# Register so P2/P3 "import LatticeCrypto as P1" works
lattice_infrastructure = types.ModuleType("lattice_infrastructure")
for name in ["LatticeParams", "gadget_matrix", "bit_decompose", "TrapGen", "discrete_gaussian", "SamplePre", "G_vector", "BinaryTreeNode", "BinaryTree", "Setup", "KeyGen"]:
    lattice_infrastructure.__dict__[name] = globals()[name]
sys.modules["LatticeCrypto"] = lattice_infrastructure
sys.modules["lattice_infrastructure"] = lattice_infrastructure

# P1 unit test
params = LatticeParams(n=64)
system = Setup(tree_depth=3, params=params)
sk = KeyGen(system, "Alice")
u = G_vector("Alice", params)
assert np.array_equal((system["A"] @ sk) % params.q, u)
print("[P1] Lattice Infrastructure: PASS", flush=True)

[P1] Lattice Infrastructure: PASS


---
## P2 — Forward Security

This section implements **forward-secure encryption** and **key evolution** (from `forward_security.py`).

### 2.1 Role in the Paper

- **Epochs:** Time is split into epochs $t = 0, 1, \ldots, T-1$ with $T = 2^d$ (tree depth $d$). Identity at epoch $t$ is $\mathsf{id}\|t$; the corresponding public vector is $\mathbf{u}_t = H(\mathsf{id}\|t)$.
- **Encrypt(user_id, epoch, bit):** Dual Regev–style IBE. For identity $\mathsf{id}$ and epoch $t$:
  - $\mathbf{u} = H(\mathsf{id}\|t)$.
  - Sample random $\mathbf{s} \in \mathbb{Z}_q^n$, small errors $\mathbf{e}_1, e_2$ (discrete Gaussian).
  - Ciphertext: $\mathbf{c}_1 = \mathbf{A}^T \mathbf{s} + \mathbf{e}_1 \bmod q$, $c_2 = \mathbf{u}^T \mathbf{s} + e_2 + (q/2)\, \mathsf{bit} \bmod q$, plus epoch $t$.
- **Decrypt(ct, key_bundle):** Uses secret key $\mathbf{sk}_t$ for the ciphertext’s epoch $t$. Computes $\mathsf{approx} = c_2 - \mathbf{sk}_t^T \mathbf{c}_1 \bmod q$; rounds to 0 or 1 depending on distance to 0 vs $q/2$.
- **Key Update / Minimal cover:** Forward security requires that at time $t$ the user only holds keys for a **minimal set of tree nodes** that cover the current and future epochs (and can drop keys for past epochs). **get_min_cover(current_time)** returns the set of node labels that form this minimal cover. **Update(key_bundle, t)** returns the key bundle and list of node indices needed for time $t+1$.
- **simulate_key_evolution(user_id, nodes):** For simulation: precomputes secret keys for given tree nodes with identities $\mathsf{user\_id}\|node$; in the real scheme these would be derived/updated via the tree and key-update algorithm.

### 2.2 Implementation Notes

- **UserOps** holds the system state (params, $\mathbf{A}$, $\mathbf{T}_{\mathbf{A}}$, tree) and provides Encrypt, Decrypt, get_min_cover, Update, and simulate_key_evolution.
- The unit test: Setup → simulate key evolution for nodes [0,1,2] → Encrypt for epoch 1 with bit 1 → Decrypt with the key bundle; asserts decryption yields 1.

In [8]:
import LatticeCrypto as P1

class UserOps:
    def __init__(self, system):
        self.params = system["params"]
        self.A = system["A"]
        self.T_A = system["T_A"]
        self.tree = system["tree"]
        self.total_epochs = 2 ** self.tree.depth

    def get_min_cover(self, current_time):
        cover = []
        def dfs(node, l, r):
            if node is None or r < current_time:
                return
            if l >= current_time:
                cover.append(node.label)
                return
            mid = (l + r) // 2
            dfs(node.left, l, mid - 1)
            if mid >= current_time:
                cover.append(node.label)
            dfs(node.right, mid + 1, r)
        dfs(self.tree.root, 0, self.total_epochs - 1)
        return sorted(set(cover))

    def Update(self, key_bundle, t):
        next_t = t + 1
        if next_t >= self.total_epochs:
            return {}, []
        needed = self.get_min_cover(next_t)
        return {k: key_bundle[k] for k in needed if k in key_bundle}, needed

    def simulate_key_evolution(self, user_id, nodes):
        bundle = {}
        for node in nodes:
            uid = f"{user_id}_{node}"
            u = P1.G_vector(uid, self.params)
            bundle[node] = P1.SamplePre(self.A, self.T_A, u, self.params)
        return bundle

    def Encrypt(self, user_id, epoch, bit):
        uid = f"{user_id}_{epoch}"
        u = P1.G_vector(uid, self.params)
        q = self.params.q
        s = np.random.randint(0, q, size=self.params.n)
        e1 = P1.discrete_gaussian((self.A.shape[1],), self.params.sigma)
        e2 = P1.discrete_gaussian((1,), self.params.sigma)[0]
        c1 = (self.A.T @ s + e1) % q
        c2 = (u @ s + e2 + (q // 2) * bit) % q
        return {"c1": c1, "c2": c2, "epoch": epoch}

    def Decrypt(self, ct, key_bundle):
        sk = key_bundle.get(ct["epoch"])
        if sk is None:
            return None
        q = self.params.q
        approx = (ct["c2"] - sk @ ct["c1"]) % q
        return 0 if min(approx, q - approx) < abs(approx - q // 2) else 1

# Register for simulation
forward_security = types.ModuleType("forward_security")
forward_security.UserOps = UserOps
sys.modules["forward_security"] = forward_security

# P2 unit test
system = P1.Setup(tree_depth=3, params=P1.LatticeParams(n=64))
ops = UserOps(system)
keys = ops.simulate_key_evolution("Alice", [0, 1, 2])
ct = ops.Encrypt("Alice", 1, 1)
assert ops.Decrypt(ct, keys) == 1
print("[P2] Forward Security: PASS", flush=True)

[P2] Forward Security: PASS


---
## P3 — Trust Model

This section implements **query authentication** and **trust/reputation** (from `Trust_Model.py`).

### 3.1 Role in the Paper

In the IoT setting, queries (e.g. search or access requests) must be **authenticated** and tied to a **trust model** so that:

- Only legitimate users can issue valid queries.
- Malicious or forged queries are rejected and can reduce the issuer’s trust score.

Components:

- **DilithiumStub:** Placeholder for a lattice-based signature scheme (e.g. Dilithium). In the paper, query messages are signed; here we use a hash-based stub (pk = H(sk), sign = H(msg\|pk), verify by recomputing). A full implementation would use NIST-standard Dilithium.
- **TrustManager:** Maintains per-user trust scores (e.g. integer; check: score ≥ 0 to allow access). **reward(uid)** increases score (capped); **penalize(uid)** decreases it (e.g. on failed verification).
- **Query:** Dataclass with `encrypted_keyword` (opaque query payload), `signature`, and `epoch` (to bind query to time).
- **QueryValidator:** Validates a query for a given user and public key: (1) **Trust check:** TrustManager allows the user. (2) **Signature check:** Message = serialize(user_id, query) is signed correctly; if not, penalize user and reject. (3) If both pass, reward user and accept. **serialize** builds the signed message as `P3|u_bytes|encrypted_keyword|epoch` where $\mathbf{u} = H(\mathsf{user\_id})$ (same hash as in P1).

### 3.2 Implementation Notes (README-aligned)

- **CheckTrust:** `TrustManager.check(uid)` is README's CheckTrust(UserID); alias `CheckTrust` is set for README naming.
- **Match Logic:** `match_query_to_data(query, encrypted_data)` implements "Compare User Query vs. Stored Data tags" (by epoch, optional tag).
- Unit test: sign/verify, `CheckTrust("Alice")`, and `match_query_to_data` (query vs list of dicts with `"epoch"`).

In [9]:
import hashlib
from dataclasses import dataclass
import LatticeCrypto as P1

# Post-Quantum Signatures (README: Dilithium-3). Stub uses SHA256; use Dilithium-3 library for full compliance.
class DilithiumStub:
    def pk_from_sk(self, sk: bytes) -> bytes:
        return hashlib.sha256(sk).digest()
    def sign(self, msg: bytes, sk: bytes) -> bytes:
        return hashlib.sha256(msg + self.pk_from_sk(sk)).digest()
    def verify(self, msg: bytes, sig: bytes, pk: bytes) -> bool:
        return hashlib.sha256(msg + pk).digest() == sig

# Trust Manager (README: TrustDatabase, CheckTrust(UserID) iff Score >= 0; penalize on bad signature)
class TrustManager:
    def __init__(self):
        self.db = {}
    def check(self, uid):
        """README: CheckTrust(UserID). Returns True if Score >= 0, else False."""
        return self.db.get(uid, 0) >= 0
    CheckTrust = check  # README naming
    def reward(self, uid):
        self.db[uid] = min(self.db.get(uid, 0) + 1, 10)
    def penalize(self, uid):
        self.db[uid] = self.db.get(uid, 0) - 1

# Query: Q = { EncryptedKeyword, Signature, Epoch_ID } (README)
@dataclass
class Query:
    """Query structure per README: Q = { EncryptedKeyword, Signature, Epoch_ID }."""
    encrypted_keyword: bytes
    signature: bytes
    epoch: int

class QueryValidator:
    def __init__(self, tm, signer, params):
        self.tm = tm
        self.signer = signer
        self.params = params
    def validate(self, user_id, q, pk):
        if not self.tm.check(user_id):
            return False
        msg = self.serialize(user_id, q)
        if not self.signer.verify(msg, q.signature, pk):
            self.tm.penalize(user_id)
            return False
        self.tm.reward(user_id)
        return True
    def serialize(self, uid, q):
        u = P1.G_vector(uid, self.params)
        return b"P3|" + u.tobytes() + q.encrypted_keyword + q.epoch.to_bytes(8, "big")

# Match Logic (README: Compare User Query vs. Stored Data tags). Returns indices of matching items.
def match_query_to_data(query: Query, encrypted_data: list) -> list:
    """Compare user query to stored data tags (epoch, optional tag). Returns indices of matches."""
    indices = []
    for i, item in enumerate(encrypted_data):
        if item.get("epoch") != query.epoch:
            continue
        if "tag" in item and item["tag"] != query.encrypted_keyword:
            continue
        indices.append(i)
    return indices

# Register for simulation
Trust_Model = types.ModuleType("Trust_Model")
Trust_Model.TrustManager = TrustManager
Trust_Model.DilithiumStub = DilithiumStub
Trust_Model.Query = Query
Trust_Model.QueryValidator = QueryValidator
Trust_Model.match_query_to_data = match_query_to_data
sys.modules["Trust_Model"] = Trust_Model

# P3 unit test (sign/verify, CheckTrust, match_query_to_data)
params = P1.LatticeParams(n=64)
tm = TrustManager()
sig = DilithiumStub()
sk, pk = b"secret", sig.pk_from_sk(b"secret")
q = Query(b"kw", b"", 0)
msg = b"P3|" + P1.G_vector("Alice", params).tobytes() + b"kw" + (0).to_bytes(8, "big")
q.signature = sig.sign(msg, sk)
v = QueryValidator(tm, sig, params)
assert v.validate("Alice", q, pk)
assert tm.CheckTrust("Alice")  # README: CheckTrust(UserID)
encrypted_data = [{"epoch": 0, "c1": None, "c2": None}, {"epoch": 1}]
assert match_query_to_data(q, encrypted_data) == [0]
assert match_query_to_data(Query(b"x", b"", 1), encrypted_data) == [1]
print("[P3] Trust Model (Sign/Verify, CheckTrust, Match): PASS", flush=True)

[P3] Trust Model (Sign/Verify, CheckTrust, Match): PASS


---
## Table 1 — fs-IBE Parameter Selection

This section reproduces **Table 1** from the paper: parameter sets for the fs-IBE scheme (from `fs_ibe_params.py`).

### Table 1 in the Paper

| Parameter   | $n$ | $q$ | $\delta$ (approx) | NIST Security (level) | bits security |
|------------|-------|-------|----------------------|------------------------|----------------|
| PARA.512   | 512   | 3329  | $2^{-139}$         | 1                      | 143            |
| PARA.768   | 768   | 3329  | $2^{-164}$         | 3                      | 207            |
| PARA.1024  | 1024  | 3329  | $2^{-174}$         | 5                      | 272            |

- **$n$:** Lattice dimension (security parameter).
- **$q$:** Modulus (here fixed to 3329, Kyber-style).
- **$\delta$:** Approximate root Hermite factor or related lattice reduction bound; smaller $\delta$ means higher security.
- **NIST level / bits security:** Post-quantum security level (NIST PQC) and equivalent “bits” of security.

**get_lattice_params(parameter_name)** returns a `LatticeParams(n, q)` for the chosen row. The cell below prints Table 1 and registers the module for P4.

In [10]:
from lattice_infrastructure import LatticeParams

FS_IBE_TABLE = [
    {"parameter": "PARA.512", "n": 512, "q": 3329, "delta_approx": "2^(-139)", "nist_level": 1, "bits_security": 143},
    {"parameter": "PARA.768", "n": 768, "q": 3329, "delta_approx": "2^(-164)", "nist_level": 3, "bits_security": 207},
    {"parameter": "PARA.1024", "n": 1024, "q": 3329, "delta_approx": "2^(-174)", "nist_level": 5, "bits_security": 272},
]

def get_lattice_params(parameter_name):
    for row in FS_IBE_TABLE:
        if row["parameter"] == parameter_name:
            return LatticeParams(n=row["n"], q=row["q"])
    raise KeyError(f"Unknown parameter: {parameter_name}")

def print_table_1():
    header = f"{'Parameter':<12} {'n':<8} {'q':<8} {'δ (approx)':<14} {'NIST Security (level)':<24} {'bits security':<16}"
    sep = "-" * 90
    lines = ["Table 1  The parameter selection of our fs-IBE.", "", header, sep]
    for row in FS_IBE_TABLE:
        lines.append(f"{row['parameter']:<12} {row['n']:<8} {row['q']:<8} {row['delta_approx']:<14} {row['nist_level']:<24} {row['bits_security']:<16}")
    text = "\n".join(lines)
    print(text, flush=True)
    return text

fs_ibe_params = types.ModuleType("fs_ibe_params")
fs_ibe_params.FS_IBE_TABLE = FS_IBE_TABLE
fs_ibe_params.get_lattice_params = get_lattice_params
fs_ibe_params.print_table_1 = print_table_1
sys.modules["fs_ibe_params"] = fs_ibe_params

print_table_1()

Table 1  The parameter selection of our fs-IBE.

Parameter    n        q        δ (approx)     NIST Security (level)    bits security   
------------------------------------------------------------------------------------------
PARA.512     512      3329     2^(-139)       1                        143             
PARA.768     768      3329     2^(-164)       3                        207             
PARA.1024    1024     3329     2^(-174)       5                        272             


'Table 1  The parameter selection of our fs-IBE.\n\nParameter    n        q        δ (approx)     NIST Security (level)    bits security   \n------------------------------------------------------------------------------------------\nPARA.512     512      3329     2^(-139)       1                        143             \nPARA.768     768      3329     2^(-164)       3                        207             \nPARA.1024    1024     3329     2^(-174)       5                        272             '

---
## P4 — Full Simulation and Benchmarking

This section runs the **full fs-IBE + trust pipeline** for all three parameter sets (PARA.512, PARA.768, PARA.1024) and writes **Results_Report.csv** (from `simulation.py`).

### 4.1 What the Simulation Does

1. **Setup:** For each parameter set (Table 1), build system (Setup), UserOps, TrustManager, DilithiumStub, QueryValidator.
2. **Data encryption:** Encrypt a small stream of data items (bits) for a fixed user and epoch; measure **data encryption time**.
3. **Query path (per query):**
   - **Query encryption time:** Time to produce an encryption (e.g. for a search token).
   - **Trust verification time:** Time to validate a signed query (TrustManager check + signature verify).
   - **Match time:** Time to find ciphertexts matching the query (here: simple epoch match over a list).
   - **Query decryption time:** Time to decrypt one relevant ciphertext.
4. **Aggregates:** **Query execution latency** = sum of the above per query; **query throughput** (queries/s); **overall model throughput** and **overall model latency** for the full run.
5. **False Trust Acceptance Rate (FTAR):** Number of queries with **wrong** signatures that are incorrectly accepted, divided by total malicious queries. For a secure validator this should be 0% (as in the unit test).

### 4.2 Output

- **Console:** Printed results for each parameter set (timing and FTAR).
- **Results_Report.csv:** One row per parameter set with columns: parameter, n, bits_security, nist_level, data_encryption_time_s, query_encryption_time_s, data_decryption_time_s, query_execution_latency_s, query_throughput_per_s, overall_model_throughput_per_s, overall_model_latency_s, false_trust_acceptance_rate, num_data, num_queries, num_malicious, malicious_accepted.

If **Results_Report.csv** is locked, the script writes **Results_Report_output.csv** instead.

In [11]:
import time
import csv
import lattice_infrastructure as P1
from forward_security import UserOps
from Trust_Model import TrustManager, DilithiumStub, Query, QueryValidator

def run_simulation(n=64, num_data=5, num_queries=10, num_malicious=5, tree_depth=3, param_name=None):
    params = P1.LatticeParams(n=n)
    system = P1.Setup(tree_depth=tree_depth, params=params)
    ops = UserOps(system)
    tm = TrustManager()
    sig = DilithiumStub()
    sk_user, pk_user = b"user_sk", sig.pk_from_sk(b"user_sk")
    validator = QueryValidator(tm, sig, params)
    user_id, epoch = "Alice", 1
    nodes = list(range(min(epoch + 2, 2 ** tree_depth)))
    keys = ops.simulate_key_evolution(user_id, nodes)

    t0 = time.perf_counter()
    encrypted_data = []
    for i in range(num_data):
        ct = ops.Encrypt(user_id, epoch, i % 2)
        encrypted_data.append(ct)
    data_encryption_time = time.perf_counter() - t0

    t_enc_q_list = []
    for _ in range(num_queries):
        t0 = time.perf_counter()
        ops.Encrypt(user_id, epoch, 1)
        t_enc_q_list.append(time.perf_counter() - t0)
    query_encryption_time = sum(t_enc_q_list) / len(t_enc_q_list) if t_enc_q_list else 0

    def one_query():
        q = Query(b"keyword", b"", epoch)
        msg = b"P3|" + P1.G_vector(user_id, params).tobytes() + q.encrypted_keyword + q.epoch.to_bytes(8, "big")
        q.signature = sig.sign(msg, sk_user)
        return q, msg

    t_trust_list = []
    for _ in range(num_queries):
        q, _ = one_query()
        t0 = time.perf_counter()
        validator.validate(user_id, q, pk_user)
        t_trust_list.append(time.perf_counter() - t0)
    trust_time = sum(t_trust_list) / len(t_trust_list) if t_trust_list else 0

    t_match_list = []
    for _ in range(num_queries):
        t0 = time.perf_counter()
        for ct in encrypted_data:
            if ct["epoch"] == epoch:
                break
        t_match_list.append(time.perf_counter() - t0)
    match_time = sum(t_match_list) / len(t_match_list) if t_match_list else 0

    t0 = time.perf_counter()
    for ct in encrypted_data:
        ops.Decrypt(ct, keys)
    data_decryption_time = time.perf_counter() - t0

    t_dec_list = []
    for _ in range(num_queries):
        t0 = time.perf_counter()
        ops.Decrypt(encrypted_data[0], keys)
        t_dec_list.append(time.perf_counter() - t0)
    query_decryption_time = sum(t_dec_list) / len(t_dec_list) if t_dec_list else 0

    t_query = query_encryption_time + trust_time + match_time + query_decryption_time
    total_query_time = t_query * num_queries
    query_throughput = num_queries / total_query_time if total_query_time > 0 else 0
    overall_latency = data_encryption_time + total_query_time + data_decryption_time
    overall_throughput = (num_data + num_queries) / overall_latency if overall_latency > 0 else 0

    malicious_accepted = 0
    for _ in range(num_malicious):
        q, _ = one_query()
        q.signature = b"wrong_signature"
        if validator.validate(user_id, q, pk_user):
            malicious_accepted += 1
    ftar = malicious_accepted / num_malicious if num_malicious > 0 else 0

    out = {"data_encryption_time_s": data_encryption_time, "query_encryption_time_s": query_encryption_time, "data_decryption_time_s": data_decryption_time, "query_execution_latency_s": t_query, "query_throughput_per_s": query_throughput, "overall_model_throughput_per_s": overall_throughput, "overall_model_latency_s": overall_latency, "false_trust_acceptance_rate": ftar, "num_data": num_data, "num_queries": num_queries, "num_malicious": num_malicious, "malicious_accepted": malicious_accepted}
    if param_name is not None:
        out["parameter"] = param_name
        out["n"] = n
    return out

def print_results(metrics, param_name=None):
    if param_name:
        print("\n" + "=" * 60, flush=True)
        print(f"  Parameter: {param_name}  (n = {metrics.get('n', '—')})", flush=True)
        print("=" * 60, flush=True)
    else:
        print("\n" + "=" * 60, flush=True)
        print("  Results (Word document & README)", flush=True)
        print("=" * 60, flush=True)
    print(f"  Data encryption time        : {metrics['data_encryption_time_s']:.6f} s", flush=True)
    print(f"  Query encryption time       : {metrics['query_encryption_time_s']:.6f} s", flush=True)
    print(f"  Data decryption time        : {metrics['data_decryption_time_s']:.6f} s", flush=True)
    print(f"  Query execution latency     : {metrics['query_execution_latency_s']:.6f} s", flush=True)
    print(f"  Query throughput            : {metrics['query_throughput_per_s']:.2f} queries/s", flush=True)
    print(f"  Overall model throughput    : {metrics['overall_model_throughput_per_s']:.2f} ops/s", flush=True)
    print(f"  Overall model latency       : {metrics['overall_model_latency_s']:.6f} s", flush=True)
    print(f"  False trust acceptance rate : {metrics['false_trust_acceptance_rate']:.2%}", flush=True)
    print("=" * 60, flush=True)

def run_all_three_parameters(num_data=5, num_queries=10, num_malicious=5, tree_depth=3):
    all_metrics = []
    for row in fs_ibe_params.FS_IBE_TABLE:
        param_name, n = row["parameter"], row["n"]
        print(f"  Running {param_name} (n={n}) ...", flush=True)
        m = run_simulation(n=n, num_data=num_data, num_queries=num_queries, num_malicious=num_malicious, tree_depth=tree_depth, param_name=param_name)
        m["bits_security"] = row["bits_security"]
        m["nist_level"] = row["nist_level"]
        all_metrics.append(m)
    return all_metrics

def print_results_all_three(all_metrics):
    for m in all_metrics:
        print_results(m, param_name=m["parameter"])

def save_csv_all_three(all_metrics, path="Results_Report.csv"):
    if not all_metrics:
        return
    fieldnames = ["parameter", "n", "bits_security", "nist_level", "data_encryption_time_s", "query_encryption_time_s", "data_decryption_time_s", "query_execution_latency_s", "query_throughput_per_s", "overall_model_throughput_per_s", "overall_model_latency_s", "false_trust_acceptance_rate", "num_data", "num_queries", "num_malicious", "malicious_accepted"]
    alt_path = "Results_Report_output.csv"
    try:
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            w.writeheader()
            for m in all_metrics:
                w.writerow(m)
        print(f"  Saved: {path}", flush=True)
    except PermissionError:
        with open(alt_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            w.writeheader()
            for m in all_metrics:
                w.writerow(m)
        print(f"  Saved: {alt_path}  (original file was in use)", flush=True)

all_metrics = run_all_three_parameters(num_data=5, num_queries=10, num_malicious=5)
print_results_all_three(all_metrics)
save_csv_all_three(all_metrics, path="Results_Report.csv")

  Running PARA.512 (n=512) ...
  Running PARA.768 (n=768) ...
  Running PARA.1024 (n=1024) ...

  Parameter: PARA.512  (n = 512)
  Data encryption time        : 0.273834 s
  Query encryption time       : 0.053237 s
  Data decryption time        : 0.000220 s
  Query execution latency     : 0.053342 s
  Query throughput            : 18.75 queries/s
  Overall model throughput    : 18.58 ops/s
  Overall model latency       : 0.807471 s
  False trust acceptance rate : 0.00%

  Parameter: PARA.768  (n = 768)
  Data encryption time        : 0.652756 s
  Query encryption time       : 0.110660 s
  Data decryption time        : 0.000237 s
  Query execution latency     : 0.110797 s
  Query throughput            : 9.03 queries/s
  Overall model throughput    : 8.52 ops/s
  Overall model latency       : 1.760960 s
  False trust acceptance rate : 0.00%

  Parameter: PARA.1024  (n = 1024)
  Data encryption time        : 3.657010 s
  Query encryption time       : 0.688622 s
  Data decryption time     

---
## Full Pipeline: Message → Encrypt → Decrypt → Result

This section runs the **complete fs-IBE flow** for a user-defined message (research-paper implementation). When you run the cell below, you will be **prompted in the terminal/console** to enter your message. After you type the message and press Enter, the notebook will encrypt it, decrypt it, and display a **result summary** with:

- **Original message** (plaintext)
- **Ciphertext** (summary: number of bits, epoch, sample of ct components)
- **Decrypted message** (recovered plaintext)
- **Parameters** (user identity, epoch, lattice parameter set)
- **Timings** (encoding, encryption, decryption)
- **Verification** (match: original vs decrypted)

In [12]:
# ========== USER INPUT: Message is read from terminal/console when you run this cell ==========
print("Enter your message (press Enter when done; leave empty for default 'Hello IoT'):", flush=True)
USER_MESSAGE = input().strip() or "Hello IoT"
print("Processing...", flush=True)

USER_ID = "Alice"
EPOCH = 1
PARAM_SET = "PARA.512"      # One of: PARA.512, PARA.768, PARA.1024 (Table 1)
TREE_DEPTH = 3

from forward_security import UserOps

# ---------- Helpers: string ↔ bits (UTF-8, 8 bits per byte) ----------
def message_to_bits(msg: str) -> list:
    """Encode message to list of bits (MSB first per byte)."""
    bits = []
    for b in msg.encode("utf-8"):
        for i in range(7, -1, -1):
            bits.append((b >> i) & 1)
    return bits

def bits_to_message(bits: list) -> str:
    """Decode list of bits to string (UTF-8)."""
    n = len(bits)
    if n % 8 != 0:
        n = (n // 8) * 8
    bytes_list = []
    for i in range(0, n, 8):
        byte_val = sum(bits[i + j] << (7 - j) for j in range(8))
        bytes_list.append(byte_val)
    return bytes(bytes_list).decode("utf-8", errors="replace")

# ---------- Setup (reuse P1, P2, Table 1) ----------
params = fs_ibe_params.get_lattice_params(PARAM_SET)
system = P1.Setup(tree_depth=TREE_DEPTH, params=params)
ops = UserOps(system)
nodes = list(range(min(EPOCH + 2, 2 ** TREE_DEPTH)))
key_bundle = ops.simulate_key_evolution(USER_ID, nodes)

# ---------- Encode message to bits ----------
bits_original = message_to_bits(USER_MESSAGE)
t0 = time.perf_counter()

# ---------- Encrypt each bit ----------
ciphertexts = []
for bit in bits_original:
    ct = ops.Encrypt(USER_ID, EPOCH, bit)
    ciphertexts.append(ct)
encrypt_time = time.perf_counter() - t0

# ---------- Decrypt each ciphertext ----------
t0 = time.perf_counter()
bits_decrypted = []
for ct in ciphertexts:
    b = ops.Decrypt(ct, key_bundle)
    bits_decrypted.append(b if b is not None else 0)
decrypt_time = time.perf_counter() - t0

# ---------- Decode to message ----------
decrypted_message = bits_to_message(bits_decrypted)

# ---------- Result summary ----------
print("\n" + "=" * 70, flush=True)
print("  FORWARD-SECURE LATTICE-BASED IBE — FULL PIPELINE RESULT (Research Paper)", flush=True)
print("=" * 70, flush=True)
print()
print("  1. ORIGINAL MESSAGE (plaintext):")
print(f"     \"{USER_MESSAGE}\"", flush=True)
print()
print("  2. ENCODING:")
print(f"     Bits (length): {len(bits_original)}  |  First 16 bits: {bits_original[:16]}", flush=True)
print()
print("  3. CIPHERTEXT (encrypted bits):")
print(f"     Number of ciphertexts: {len(ciphertexts)}  (one per bit)", flush=True)
print(f"     Epoch: {EPOCH}  |  User identity: \"{USER_ID}\"", flush=True)
if ciphertexts:
    # First ciphertext in full (c1: first 20 + ... + last 20 to keep output readable)
    ct0 = ciphertexts[0]
    c1, c2, ep = ct0["c1"], ct0["c2"], ct0["epoch"]
    k = 20
    if len(c1) <= 2 * k:
        c1_preview = list(c1)
    else:
        c1_preview = list(c1[:k]) + ["..."] + list(c1[-k:])
    print(f"     First ciphertext (bit 0): c1 = {c1_preview}  (length={len(c1)}), c2 = {c2}, epoch = {ep}", flush=True)
    print(f"     All ciphertexts (bit index, c2, epoch):", flush=True)
    for i, ct in enumerate(ciphertexts):
        print(f"       ct[{i}]: c2={ct['c2']}, epoch={ct['epoch']}", flush=True)
    # Save full ciphertext to file so nothing is truncated
    import json
    ct_export = [{"c1": ct["c1"].tolist(), "c2": int(ct["c2"]), "epoch": ct["epoch"]} for ct in ciphertexts]
    out_path = "ciphertext_output.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(ct_export, f, indent=0)
    print(f"     Full ciphertext (all c1, c2, epoch) saved to: {out_path}", flush=True)
print()
print("  4. DECRYPTED MESSAGE (recovered plaintext):")
print(f"     \"{decrypted_message}\"", flush=True)
print()
print("  5. PARAMETERS (Table 1):")
print(f"     Parameter set: {PARAM_SET}  |  n={params.n}, q={params.q}", flush=True)
print(f"     Tree depth: {TREE_DEPTH}  |  Epoch: {EPOCH}", flush=True)
print()
print("  6. TIMINGS:")
print(f"     Encryption (all bits): {encrypt_time:.6f} s", flush=True)
print(f"     Decryption (all bits): {decrypt_time:.6f} s", flush=True)
print()
print("  7. VERIFICATION:")
match = USER_MESSAGE == decrypted_message
print(f"     Original == Decrypted: {match}  " + ("[OK]" if match else "[MISMATCH]"), flush=True)
print()
print("=" * 70, flush=True)

Enter your message (press Enter when done; leave empty for default 'Hello IoT'):
Processing...

  FORWARD-SECURE LATTICE-BASED IBE — FULL PIPELINE RESULT (Research Paper)

  1. ORIGINAL MESSAGE (plaintext):
     "enter"

  2. ENCODING:
     Bits (length): 40  |  First 16 bits: [0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0]

  3. CIPHERTEXT (encrypted bits):
     Number of ciphertexts: 40  (one per bit)
     Epoch: 1  |  User identity: "Alice"
     First ciphertext (bit 0): c1 = [295, 2453, 2452, 2092, 2919, 1214, 2749, 2091, 774, 2733, 1302, 746, 1546, 2458, 1644, 2152, 3320, 84, 3099, 2578, '...', 1659, 3322, 0, 3327, 3322, 3313, 3297, 3263, 1673, 16, 30, 50, 105, 204, 416, 832, 1662, 3328, 3324, 3319]  (length=12288), c2 = 2006, epoch = 1
     All ciphertexts (bit index, c2, epoch):
       ct[0]: c2=2006, epoch=1
       ct[1]: c2=2421, epoch=1
       ct[2]: c2=1678, epoch=1
       ct[3]: c2=2383, epoch=1
       ct[4]: c2=1172, epoch=1
       ct[5]: c2=1704, epoch=1
       ct[6]: c2

---
## Summary and Reproducibility

This notebook implements the **forward-secure lattice-based IBE scheme** described in *A lattice-based forward secure IBE scheme for Internet of things*.

- **P1–P3** provide the building blocks (lattice infrastructure, forward-secure encrypt/decrypt and key update, trust model and query validation).
- **Table 1** defines the three parameter sets used in the paper (PARA.512, PARA.768, PARA.1024).
- **P4** runs the full simulation and produces **Results_Report.csv**, which can be used to reproduce or extend the paper’s performance and security (FTAR) figures.

**Reproducibility:** Run all cells in order from top to bottom. Results depend on the chosen $n$ and the number of data items, queries, and malicious queries in `run_all_three_parameters(...)`; adjust those arguments to match the paper’s experimental setup if needed.